# Lesson 02 · Empirical Bayes estimation

**Source:** *Introduction to Empirical Bayes: Examples from Baseball Statistics*, David Robinson (2017), Chapter 3 (pp. 20–27)

**Question:** Which is better, 4 hits in 10 at-bats or 300 in 1,000? Who were the best and worst hitters in history, when some players batted once and others ten thousand times?

**Goal:** Estimate a beta prior from the whole dataset by maximum likelihood, then shrink each player's average toward it.

**Contents**

1. Build the career batting table
2. Why raw averages fail
3. Step 1: estimate a prior from all the data
4. Step 2: shrink each player toward the prior
5. Results: the best and worst batters
6. Seeing shrinkage
7. Save your prior for later lessons
8. Exercises

**Tags:** `empirical-bayes` `beta-binomial` `maximum-likelihood` `shrinkage` `scipy-optimize` `pandas-joins`

**Before you start:** Lesson 01. The first run downloads about 10 MB of data into `data/lahman/`.

*How this notebook works:* each **Your turn** prompt is followed by an empty cell for your code. **Check** boxes give values to compare against; they come from the book, and your numbers can differ slightly because the Lahman data now runs through more recent seasons. **Reflect** prompts are for short written answers. Every prompt is numbered *lesson.section.question* (for example **2.3.1**), so you can refer to it; empty code cells and answer cells carry the same number.

In [ ]:
# --- Setup: run this cell first ------------------------------------------
# Windows + conda: put the environment's DLL folders on PATH, so plotting
# can't crash the kernel when Jupyter wasn't started from the activated env.
import os, sys
for sub in ["Library\\mingw-w64\\bin", "Library\\usr\\bin", "Library\\bin", "Scripts", "bin"]:
    p = os.path.join(sys.prefix, sub)
    if os.path.isdir(p) and p not in os.environ["PATH"]:
        os.environ["PATH"] = p + os.pathsep + os.environ["PATH"]

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy
from scipy import stats, optimize, special

import eb_data   # helpers in this folder: loading the Lahman data, saving results

rng = np.random.default_rng(2017)
pd.set_option("display.precision", 4)
plt.rcParams["figure.figsize"] = (7, 4)
print(f"numpy {np.__version__} | pandas {pd.__version__} | scipy {scipy.__version__}")

# Download the Lahman tables the first time (skips files you already have)
eb_data.download_lahman()
batting = eb_data.load_table("Batting")
pitching = eb_data.load_table("Pitching")
people = eb_data.load_table("People")
print(f"Batting: {len(batting):,} rows | Pitching: {len(pitching):,} | People: {len(people):,}")

## 1. Build the career batting table

Each row of `Batting` is one player's season with one team (a *stint*). We want one row per player, with career hits `H` and at-bats `AB`. In this chapter the book drops **anyone who ever pitched**, because pitchers are much weaker hitters and form a different population. (We'll come back to them in Lesson 08.)

**2.1.1 · Your turn.** Build `career` with columns `playerID, name, H, AB, average`:
1. keep rows with `AB > 0`;
2. drop every `playerID` that appears in `pitching` (an *anti-join*: `~batting["playerID"].isin(...)`);
3. group by player and sum `H` and `AB`;
4. add `average = H / AB`;
5. merge in names from `people` (`nameFirst + " " + nameLast`).

In [ ]:
# 2.1.1 · Your code here

**2.1.2 · Your turn.** Compare your table with `eb_data.career(pitcher_rule=None)`, which follows the same recipe. Do the row counts and totals match?

In [ ]:
# 2.1.2 · Your code here

> **Check:** the book had 9,342 players. Expect a few hundred more with recent seasons.

## 2. Why raw averages fail

**2.2.1 · Your turn.** Show the 5 players with the highest raw average and the 5 with the lowest. Include `H` and `AB`.

In [ ]:
# 2.2.1 · Your code here

**2.2.2 · Reflect.** What do the top and bottom lists have in common? One common fix is a minimum-AB filter. What's wrong with that fix?

*2.2.2 · Your answer:*

## 3. Step 1: estimate a prior from all the data

We model each player's true average as a draw from one beta distribution, and his hits as binomial given that average:

$$p_i \sim \text{Beta}(\alpha_0, \beta_0), \qquad H_i \sim \text{Binomial}(AB_i, p_i).$$

Averaging over the unknown $p_i$ gives the **beta-binomial** distribution for $H_i$. We choose the *hyperparameters* $\alpha_0, \beta_0$ that maximize the beta-binomial likelihood of the observed hits. In SciPy that's `stats.betabinom.logpmf(H, AB, a, b)`.

Following the book, fit the prior only on players with **more than 500 at-bats**, where the averages are less noisy. (Lesson 06 shows why this filter is a crude fix.)

**2.3.1 · Your turn.** Plot a histogram of `average` for players with `AB > 500`. Does a single-peaked beta look plausible?

In [ ]:
# 2.3.1 · Your code here

**2.3.2 · Your turn.** Write `neg_log_lik(params, H, AB)` returning the negative summed beta-binomial log-likelihood, then minimize it with `optimize.minimize`.

Tip: optimize over `log(alpha), log(beta)` so the parameters stay positive without bounds. Start from something like `alpha = 1, beta = 10`.

In [ ]:
# 2.3.2 · Your code here

> **Check:** α₀ ≈ 101.4 and β₀ ≈ 287.3, so the prior mean α₀/(α₀+β₀) ≈ 0.261.

**2.3.3 · Your turn.** Overlay the fitted Beta(α₀, β₀) density on the histogram (use `density=True`).

In [ ]:
# 2.3.3 · Your code here

**2.3.4 · Your turn.** A quicker alternative is the method of moments: match the mean and variance of `average` (for AB > 500) to the beta formulas. Compute it and compare with your MLE. Why might they differ?

In [ ]:
# 2.3.4 · Your code here

## 4. Step 2: shrink each player toward the prior

Now treat the fitted Beta($\alpha_0$, $\beta_0$) as everyone's prior, and update with each player's own record. The posterior mean is the **empirical Bayes estimate**:

$$\hat p_i = \frac{H_i + \alpha_0}{AB_i + \alpha_0 + \beta_0}.$$

**2.4.1 · Your turn.** Compute the estimate for a 300/1000 hitter and a 4/10 hitter.

In [ ]:
# 2.4.1 · Your code here

> **Check:** about 0.289 for 300/1000 and 0.264 for 4/10, so the 300/1000 hitter now ranks higher.

**2.4.2 · Your turn.** Add an `eb_estimate` column to `career` for every player.

In [ ]:
# 2.4.2 · Your code here

## 5. Results: the best and worst batters

**2.5.1 · Your turn.** Show the top 5 and bottom 5 players by `eb_estimate`, with `H`, `AB`, `average` and `eb_estimate`.

In [ ]:
# 2.5.1 · Your code here

> **Check:** Rogers Hornsby should top the list (about .354). Bill Bergen should be near the bottom.

## 6. Seeing shrinkage

**2.6.1 · Your turn.** Scatter `average` (x) against `eb_estimate` (y), coloured by `AB` on a log scale (`matplotlib.colors.LogNorm`). Add the line y = x and a horizontal line at the prior mean.

In [ ]:
# 2.6.1 · Your code here

**2.6.2 · Reflect.** Which points barely move, and which move a lot? Put the rule into one sentence.

*2.6.2 · Your answer:*

## 7. Save your prior for later lessons

Later lessons start from this prior. `eb_data.load_prior()` reads it back, and falls back to the book's values (101.4, 287.3) if you skip this step.

**2.7.1 · Your turn.** Save it: `eb_data.save_result("prior", {"alpha0": float(alpha0), "beta0": float(beta0)})`.

In [ ]:
# 2.7.1 · Your code here

## 8. Exercises

**2.8.1 · Your turn.** How sensitive is the prior to the 500-AB cutoff? Refit with cutoffs of 100, 250, 1000 and 2000, and tabulate α₀, β₀ and the prior mean. What pattern do you see, and why might it happen? (Lesson 06 explains it.)

In [ ]:
# 2.8.1 · Your code here

**2.8.2 · Your turn.** **Card connection.** Pokémon cards are released in *sets* (a new one comes out every few months), and each set is sold in sealed *booster packs* of about ten randomly assorted cards. Now and then a pack contains a rare, valuable card, which collectors call a *hit*. The chance that a pack contains a hit is the set's *pull rate*, and it differs from set to set. Collectors estimate pull rates by logging the packs they open, so a popular set may have hundreds of packs logged and an obscure one only a handful: the same situation as players with many or few at-bats.

Treat each set as a "player", with packs opened as at-bats and hits as hits. Simulate 60 sets whose true pull rates come from Beta(4, 76) (about 1 hit in 20 packs on average), with packs opened ranging from 3 to 400 (`rng.integers`). Fit a prior to the simulated counts, shrink, and plot raw rate against shrunken rate. Which sets move most?

In [ ]:
# 2.8.2 · Your code here

## References

- *Introduction to Empirical Bayes: Examples from Baseball Statistics*, David Robinson (2017), Chapter 3.
- Lahman Baseball Database (CC BY-SA 3.0): https://sabr.org/lahman-database/
- SciPy `betabinom`: https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.betabinom.html
- Efron, *Large-Scale Inference* (2010), Chapter 1, for the empirical Bayes idea in general.